# 📘 Deep Learning Text Generation Learning Project
## Text Generation using **Vanilla RNN, LSTM, and GRU**

This notebook is built for **students and beginners** to understand how sequence models learn:
- grammar
- sentence flow
- contextual dependencies
- next-word prediction
- text generation

🎯 **Goal:** Compare **Simple RNN vs LSTM vs GRU** on the same text corpus and understand why gated architectures perform better.

# 🧠 Problem Statement
Design and implement a DL model capable of learning the underlying structure, grammar, and contextual dependencies of a given text corpus to generate coherent and meaningful text sequences using:

1. **Vanilla RNN**
2. **LSTM**
3. **GRU**

Then compare:
- training loss
- generated text quality
- memory handling
- long-term dependency learning

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import numpy as np
import matplotlib.pyplot as plt
print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


# 📥 Load Text Corpus
We use a **small built-in sample corpus** so students can run this quickly.
You can later replace it with:
- Shakespeare text
- song lyrics
- chatbot data
- story paragraphs
- custom PDF extracted text

In [ ]:
corpus = '''
deep learning is transforming artificial intelligence
recurrent neural networks are useful for sequential data
lstm helps remember long term dependencies
gru is faster and simpler than lstm
text generation models predict the next word
deep learning models can generate meaningful sentences
'''
print(corpus)

# 🔤 Tokenization & Sequence Creation
We convert text into integer tokens and create **n-gram style sequences**
for next-word prediction.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

# 🧠 Model 1: Vanilla RNN
This is the baseline sequential model.
It struggles with long-term dependencies because of vanishing gradients.

In [ ]:
rnn_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    SimpleRNN(64),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

rnn_history = rnn_model.fit(X, y, epochs=100, verbose=0)
print("Vanilla RNN training completed")

# 🔒 Model 2: LSTM
LSTM uses **input, forget, and output gates**
to preserve long-term memory.

In [ ]:
lstm_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

lstm_history = lstm_model.fit(X, y, epochs=100, verbose=0)
print("LSTM training completed")

# ⚡ Model 3: GRU
GRU uses **reset + update gates**.
It is computationally faster than LSTM and often gives similar results.

In [ ]:
gru_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    GRU(64),
    Dense(total_words, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

gru_history = gru_model.fit(X, y, epochs=100, verbose=0)
print("GRU training completed")

## 📉 Compare Training Loss

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(rnn_history.history['loss'], label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'], label='GRU')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

# ✍️ Text Generation Function
This function predicts the next word repeatedly to generate a sentence.

In [ ]:
def generate_text(model, seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

## 🧪 Generate Text Samples

In [ ]:
print("RNN :", generate_text(rnn_model, "deep learning", 5))
print("LSTM:", generate_text(lstm_model, "deep learning", 5))
print("GRU :", generate_text(gru_model, "deep learning", 5))

# 📚 Student Learning Tasks
### ✅ Beginner Tasks
1. Replace corpus with your own paragraph
2. Increase embedding dimension
3. Increase epochs to 200
4. Change hidden units 64 → 128
5. Generate 10 words instead of 5

# ✅ Conclusion
- **Vanilla RNN** learns short patterns but struggles with memory
- **LSTM** captures long-range grammar dependencies better
- **GRU** gives similar performance with fewer gates and faster training
- This notebook helps students understand **sequence modeling mathematically and practically**

# 📥 Task 1 — Custom Paragraph Corpus

The original corpus is replaced with a custom paragraph focused on **Space Exploration and Technology**. This helps the models learn meaningful word relationships and generate more relevant and unique text.

In [ ]:
new_corpus = """
space exploration has expanded human knowledge of the universe
rocket engines generate thrust to escape earth gravity
satellites orbit earth to provide communication and navigation
astronauts conduct experiments aboard the international space station
mars rovers collect soil samples and send images back to earth
telescopes capture images of distant galaxies and nebulae
artificial intelligence assists scientists in analyzing space data
future missions aim to land humans on mars within this decade
"""
print(new_corpus)

# 🔤 Tokenization & N-gram Sequence Creation

The custom corpus is tokenized using the Keras Tokenizer, and each sentence is converted into progressive n-gram sequences. These sequences are padded to a fixed length using pad_sequences and then divided into input (X) and target (y) data for training.

In [ ]:
# Tokenization on custom corpus
tokenizer = Tokenizer()
tokenizer.fit_on_texts([new_corpus])

In [ ]:
print("Integer Encoded Tokens of the Custom Corpus:")
tokenizer.word_index

In [ ]:
total_words = len(tokenizer.word_index) + 1

# Create n-gram sequences
input_sequences = []
for line in new_corpus.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

# Padding
max_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_len, padding='pre')
)

# Split into X and y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Vocabulary size:", total_words)
print("Max sequence length (max_len):", max_len)
print("X shape:", X.shape)
print("y shape:", y.shape)

# ⚙️ Tasks 2, 3, 4 & 5 — Model Configuration Updates

The model settings were updated by increasing the embedding dimension from **32** to **64**, hidden units from **64** to **128**, and training epochs from **100** to **200**. The text generation function was also modified to predict **10** words instead of **5**. These updated settings are applied consistently across the Vanilla RNN, LSTM, and GRU models for a fair performance comparison.

In [ ]:
Embedding_size = 64
Epochs = 200
Hidden_Units = 128
No_of_Words = 10

## 🧠 Vanilla RNN with Updated Configuration

In [ ]:
rnn = Sequential([
    Embedding(input_dim=total_words, output_dim=Embedding_size, input_length=max_len-1),
    SimpleRNN(Hidden_Units),
    Dense(total_words, activation='softmax')
])

rnn.compile(loss='sparse_categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

rnn_history = rnn.fit(X, y, epochs=Epochs, verbose=0)

print("Customized RNN")
print("Final Accuracy:", rnn_history.history['accuracy'][-1])
print("Final Loss:", rnn_history.history['loss'][-1])

## 🧠 LSTM with Updated Configuration

In [ ]:
lstm = Sequential([
    Embedding(input_dim=total_words, output_dim=Embedding_size, input_length=max_len-1),
    LSTM(Hidden_Units),
    Dense(total_words, activation='softmax')
])

lstm.compile(loss='sparse_categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

lstm_history = lstm.fit(X, y, epochs=Epochs, verbose=0)

print("Customized LSTM")
print("Final Accuracy:", lstm_history.history['accuracy'][-1])
print("Final Loss:", lstm_history.history['loss'][-1])

## 🧠 GRU with Updated Configuration

In [ ]:
gru = Sequential([
    Embedding(total_words, Embedding_size, input_length=max_len-1),
    GRU(Hidden_Units),
    Dense(total_words, activation='softmax')
])

gru.compile(loss='sparse_categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

gru_history = gru.fit(X, y, epochs=Epochs, verbose=0)

print("Customized GRU")
print("Final Accuracy:", gru_history.history['accuracy'][-1])
print("Final Loss:", gru_history.history['loss'][-1])

# 📉 Training Loss Comparison

The training loss of the **Customized Vanilla RNN**, **Customized LSTM**, and **Customized GRU** models is plotted over **200 epochs** to compare their learning progress during training. The plot helps visualize how each model's loss decreases and converges on the custom corpus.

In [ ]:
import seaborn as sns
plt.figure(figsize=(11, 5))
sns.lineplot(rnn_history.history['loss'], label='Customized RNN', color='purple')
sns.lineplot(lstm_history.history['loss'], label='Customized LSTM', color='orange')
sns.lineplot(gru_history.history['loss'], label='Customized GRU', color='blue')
plt.xlabel("No of Epochs")
plt.ylabel("Cross-Entropy Loss")
plt.title("Customized Models — Training Loss over 200 Epochs\n(Embedding dim=64, No. of Hidden States=128)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Task 5 — Generate 10 Words Per Prompt

The trained models are used to generate **10 consecutive words** from the seed phrase **"space exploration"**. The same text generation function is applied to the Vanilla RNN, LSTM, and GRU models, allowing their generated outputs to be compared using the same starting context.

In [ ]:
def generate_text_v2(model, seed_text, next_words=10):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

shared_seed = "space exploration"

print("RNN  :", generate_text_v2(rnn, shared_seed, No_of_Words))
print("LSTM :", generate_text_v2(lstm, shared_seed, No_of_Words))
print("GRU  :", generate_text_v2(gru, shared_seed, No_of_Words))

# ✅ Part 2 — Task Completion Summary
* Replaced the original corpus with a custom paragraph focused on **Space Exploration and Technology**.
* Increased the embedding dimension from **32** to **64** for richer word embeddings.
* Increased the training epochs from **100** to **200** to improve learning.
* Increased the hidden units from **64** to **128** for the Vanilla RNN, LSTM, and GRU models.
* Modified the text generation function to generate **10** words instead of **5** using the seed text **space exploration**.

## 🔍 Observations
* All three models learned the training sequences effectively and achieved high training accuracy on the custom corpus.
* The increased embedding size and hidden units helped the models capture richer word relationships and contextual information.
* Training for **200** epochs resulted in a gradual reduction in loss and improved convergence across all three architectures.
* Since the dataset contains only a small number of sentences, the models primarily memorized the training patterns, leading to very high accuracy.
* The generated text followed the structure and vocabulary of the custom corpus, producing meaningful continuations with occasional word repetitions due to the limited training data.
* **LSTM** and **GRU** generated more coherent long-word sequences than the **Vanilla RNN**, demonstrating their ability to better capture sequential dependencies.